# PDF Document Processing and TF-IDF

This project extracts text from a collection of PDF documents and represents each document using manually implemented Term Frequency (TF), Inverse Document Frequency (IDF), and TF-IDF.

## Workflow
1. Load PDF files from a configurable folder.
2. Extract text from every PDF page.
3. Convert text to lowercase and clean it.
4. Tokenize and remove English stopwords.
5. Calculate TF for each document.
6. Calculate IDF across the document collection.
7. Calculate TF-IDF weights and construct document vectors.
8. Display the top-ranked terms and export the results.

**The TF, IDF, and TF-IDF calculations are implemented manually rather than using `TfidfVectorizer`.**

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import re
import math
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import PyPDF2
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

PDF_DIR = Path("pdf_documents")
OUTPUT_DIR = Path("outputs")
TOP_N = 10

## 2. Prepare NLTK resources

The following NLTK resources are used for word tokenization and English stopword removal.

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

english_stopwords = set(stopwords.words("english"))
print("NLTK resources are ready.")

## 3. Load PDF documents and extract text

Create a folder named `pdf_documents` beside this notebook and place your PDF files inside it. The notebook reads all `.pdf` files in that folder.

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract text from every page in one PDF file."""
    page_texts = []

    try:
        with pdf_path.open("rb") as file:
            reader = PyPDF2.PdfReader(file)

            if reader.is_encrypted:
                try:
                    reader.decrypt("")
                except Exception:
                    print(f"Skipping encrypted PDF: {pdf_path.name}")
                    return ""

            for page in reader.pages:
                page_texts.append(page.extract_text() or "")

    except Exception as error:
        print(f"Could not read {pdf_path.name}: {error}")
        return ""

    return "\\n".join(page_texts).strip()


if not PDF_DIR.exists():
    raise FileNotFoundError(
        f"Folder not found: {PDF_DIR.resolve()}. "
        "Create pdf_documents and add your PDF files."
    )

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {PDF_DIR.resolve()}. "
        "Add at least one PDF file and run this cell again."
    )

documents = []
document_names = []

for pdf_file in pdf_files:
    extracted_text = extract_text_from_pdf(pdf_file)
    if extracted_text:
        documents.append(extracted_text)
        document_names.append(pdf_file.name)
    else:
        print(f"No extractable text found in: {pdf_file.name}")

if not documents:
    raise ValueError(
        "No text could be extracted from the supplied PDFs. "
        "They may be scanned/image-only PDFs; OCR would be required."
    )

print(f"Successfully extracted text from {len(documents)} PDF files.")
display(pd.DataFrame({
    "document": document_names,
    "characters": [len(text) for text in documents]
}))

## 4. Lowercase and clean the text

In [ ]:
def clean_text(text):
    """Lowercase text, remove digits and non-English characters, normalize spaces."""
    text = text.lower()
    text = re.sub(r"\\d+", " ", text)
    text = re.sub(r"[^a-zA-Z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text)
    return text.strip()


clean_documents = [clean_text(text) for text in documents]

print("Example cleaned text:")
print(clean_documents[0][:1000])

## 5. Tokenization and stopword removal

In [ ]:
tokenized_documents = [word_tokenize(text) for text in clean_documents]

processed_documents = [
    [word for word in tokens if word not in english_stopwords]
    for tokens in tokenized_documents
]

# Remove documents that become empty after preprocessing.
valid_indices = [
    index for index, tokens in enumerate(processed_documents)
    if len(tokens) > 0
]

if not valid_indices:
    raise ValueError(
        "All documents became empty after preprocessing and stopword removal."
    )

document_names = [document_names[i] for i in valid_indices]
documents = [documents[i] for i in valid_indices]
clean_documents = [clean_documents[i] for i in valid_indices]
tokenized_documents = [tokenized_documents[i] for i in valid_indices]
processed_documents = [processed_documents[i] for i in valid_indices]

print("Number of processed documents:", len(processed_documents))
for name, tokens in zip(document_names, processed_documents):
    print(f"{name}: {len(tokens)} tokens")

## 6. Term Frequency (TF)

In [ ]:
tf_documents = []

for tokens in processed_documents:
    word_counts = Counter(tokens)
    total_words = len(tokens)

    tf = {
        word: count / total_words
        for word, count in word_counts.items()
    }
    tf_documents.append(tf)

print("TF example (first document):")
print(list(tf_documents[0].items())[:15])

## 7. Inverse Document Frequency (IDF)

In [ ]:
number_of_documents = len(processed_documents)
document_frequency = Counter()

for tokens in processed_documents:
    document_frequency.update(set(tokens))

idf = {
    word: math.log(number_of_documents / frequency)
    for word, frequency in document_frequency.items()
}

print("Number of unique vocabulary terms:", len(idf))
print("IDF examples:", list(idf.items())[:15])

## 8. Calculate TF-IDF

The original assignment uses the formula:

\[
TFIDF(t,d) = TF(t,d) \times \log\left(\frac{N}{DF(t)}\right)
\]

where `N` is the number of documents and `DF(t)` is the number of documents containing term `t`.

In [ ]:
tfidf_documents = []

for tf in tf_documents:
    tfidf = {
        word: term_frequency * idf[word]
        for word, term_frequency in tf.items()
    }
    tfidf_documents.append(tfidf)

print("TF-IDF example (first document):")
print(sorted(tfidf_documents[0].items(), key=lambda item: item[1], reverse=True)[:15])

## 9. Build the document-term matrix

In [ ]:
vocabulary = sorted(idf.keys())

tfidf_matrix = np.zeros(
    (len(tfidf_documents), len(vocabulary)),
    dtype=np.float64
)

word_to_index = {
    word: index
    for index, word in enumerate(vocabulary)
}

for document_index, tfidf in enumerate(tfidf_documents):
    for word, score in tfidf.items():
        tfidf_matrix[document_index, word_to_index[word]] = score

tfidf_df = pd.DataFrame(
    tfidf_matrix,
    index=document_names,
    columns=vocabulary
)

print("Document-term matrix shape:", tfidf_df.shape)
display(tfidf_df.iloc[:, :min(15, len(vocabulary))])

## 10. Display top TF-IDF terms per document

In [ ]:
top_terms_records = []

for document_name, tfidf in zip(document_names, tfidf_documents):
    top_terms = sorted(
        tfidf.items(),
        key=lambda item: item[1],
        reverse=True
    )[:TOP_N]

    print(f"\\n{document_name} — Top {TOP_N} terms")
    for rank, (word, score) in enumerate(top_terms, start=1):
        print(f"{rank:>2}. {word:<20} {score:.6f}")
        top_terms_records.append({
            "document": document_name,
            "rank": rank,
            "term": word,
            "tfidf_score": score
        })

top_terms_df = pd.DataFrame(top_terms_records)

## 11. Visualize top terms

In [ ]:
for document_name, tfidf in zip(document_names, tfidf_documents):
    top_terms = sorted(
        tfidf.items(),
        key=lambda item: item[1],
        reverse=True
    )[:TOP_N]

    if not top_terms:
        continue

    words = [item[0] for item in top_terms][::-1]
    scores = [item[1] for item in top_terms][::-1]

    plt.figure(figsize=(8, 5))
    plt.barh(words, scores)
    plt.xlabel("TF-IDF score")
    plt.title(f"Top {TOP_N} TF-IDF terms — {document_name}")
    plt.tight_layout()
    plt.show()

## 12. Export results

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tfidf_df.to_csv(OUTPUT_DIR / "tfidf_document_term_matrix.csv")
top_terms_df.to_csv(OUTPUT_DIR / "top_tfidf_terms.csv", index=False)

document_summary = pd.DataFrame({
    "document": document_names,
    "raw_characters": [len(text) for text in documents],
    "tokens_after_preprocessing": [len(tokens) for tokens in processed_documents],
    "unique_terms": [len(set(tokens)) for tokens in processed_documents]
})
document_summary.to_csv(OUTPUT_DIR / "document_summary.csv", index=False)

print(f"Saved output files to: {OUTPUT_DIR.resolve()}")
print(" - tfidf_document_term_matrix.csv")
print(" - top_tfidf_terms.csv")
print(" - document_summary.csv")

## Notes and limitations

- The notebook processes text-based PDFs. Scanned PDFs that contain only images require OCR, which is not included here.
- Text extraction quality depends on the PDF's internal text layer and layout.
- The cleaning pipeline keeps English letters only and removes numbers and punctuation, matching the original assignment's approach.
- IDF uses the unsmoothed formula `log(N / DF)`. A term appearing in every document receives an IDF score of zero.
- This notebook implements TF, IDF, and TF-IDF manually. It does not use a prebuilt TF-IDF vectorizer.
- Generated CSV files are saved in `outputs/` and can be committed to GitHub if you want to publish sample results.